# US Beta / Required Return — Batch Runner V3

**저장 테이블** : `us_required_return_result`  
**저장 형태**   : `date · ticker · indicator · value` (long-format)  
**저장 방법**   : `INSERT ... ON DUPLICATE KEY UPDATE` (upsert)  
**PRIMARY KEY** : `(date, ticker, indicator)`

---

| 셀 | 단계 |
|----|------|
| 1  | 환경 설정 & 경로 자동 감지 (노트북/데스크탑 자동 판별) |
| 2  | 모듈 Import & 설정 상수 |
| 3  | DB 연결 함수 / 조회 함수 |
| 4  | FMP / FDR 가격 fetch 함수 |
| 5  | RF(국채금리) fetch 함수 |
| 6  | Beta · Required Return 계산 함수 |
| 7  | MySQL 저장 함수 (sanitize + upsert) |
| 8  | 가격 확보 함수 (ensure_price_series) |
| 9  | 티커 1개 업데이트 함수 (update_one_ticker) |
| 10 | SPY / RF 공통 데이터 준비 |
| 11 | 배치 실행 (구간 지정 · 체크포인트 · 실패 재시도) |
| 12 | DB 조회 유틸 (pivot 조회) |
| 13 | 결과 조회 테스트 |

---
### 개선 사항 (V2 → V3)
- **경로 자동 감지**: 노트북/데스크탑 환경 자동 판별, `config.py` 경유 DB 접속
- **Python 3.9 호환**: `X | Y` 타입 힌트 제거 → `Optional[X]` 방식 통일
- **구간 실행**: `TICKER_START` / `TICKER_END` 로 2000개 티커를 단계적으로 처리
- **진행률 표시**: `[idx/total] (pct%)  ticker` 형태로 실시간 출력
- **메모리 관리**: 1개 티커 처리 후 즉시 저장 + `gc.collect()` (배치 누적 방지)
- **로그 통일**: `[HH:MM:SS][TAG] msg` 형태
- **FDR fallback 강화**: FMP 실패 시 FDR 자동 전환

## Cell 1 · 환경 설정 & 경로 자동 감지

In [1]:
import sys, os, gc, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── 후보 프로젝트 루트 (노트북 / 데스크탑) ───────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",         # 노트북
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",  # 데스크탑
]

def _setup_path() -> str:
    """
    DATA/ 폴더를 포함하는 프로젝트 루트를 탐색해 sys.path 에 추가합니다.
    탐색 순서:
      1) cwd / __file__ 상위 경로 중 DATA/ 를 포함하는 첫 번째 경로
      2) _CANDIDATE_ROOTS 에서 DATA/ 가 있는 첫 번째 경로
    """
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()

    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root

    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate) and os.path.isdir(os.path.join(candidate, "DATA")):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root 후보 경로 : {candidate}")
            return candidate

    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다.\n"
        "_CANDIDATE_ROOTS 를 현재 환경에 맞게 수정하거나 "
        "노트북을 프로젝트 루트 아래에서 실행하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")
print(f"[확인] DATA 경로    : {os.path.join(_ROOT, 'DATA')}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] DATA 경로    : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA


## Cell 2 · 모듈 Import & 설정 상수

In [2]:
# ── 표준 라이브러리 ───────────────────────────────────────────
import math
import time
import requests
from datetime import datetime
from typing import Optional, Dict, Any, List, Tuple
from pandas.tseries.offsets import BDay

# ── 외부 라이브러리 ───────────────────────────────────────────
import numpy as np
import pandas as pd
import pymysql
import FinanceDataReader as fdr
from IPython.display import display

# ── 내부 모듈 ─────────────────────────────────────────────────
from DATA.config import get_db_info, get_engine
from DATA.us_target_ticker_list_2000 import ticker_list as DEFAULT_TICKER_LIST

# ── 로그 유틸 ─────────────────────────────────────────────────
def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}")

# ══════════════════════════════════════════════════════════════
#  설정 상수 — 여기만 수정하세요
# ══════════════════════════════════════════════════════════════
DB_NAME       = "investar"
TABLE_RESULT  = "us_required_return_result"
DEFAULT_PORT  = 3307
MARKET_TICKER = "SPY"

# FMP API
API_KEY            = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"   # ← 필요시 변경
MAX_RETRY          = 5
SLEEP_BETWEEN_CALLS = 0.35
MIN_START_DATE     = "2015-01-01"

# Beta 윈도우
BETA_WINDOWS         = [252, 750, 1250]   # beta_252, beta_750, beta_1250
MAX_ROLLING_WINDOW   = max(BETA_WINDOWS)  # 1250
ROLLING_WARMUP_BDAYS = MAX_ROLLING_WINDOW + 80

# 저장 모드: "minimal" (Re + beta) / "full" (모든 중간값 포함)
STORE_MODE = "minimal"

# 체크포인트
CHECKPOINT_DIR    = "_batch_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DEFAULT_DONE_PATH = os.path.join(CHECKPOINT_DIR, "done_tickers.txt")
DEFAULT_FAIL_PATH = os.path.join(CHECKPOINT_DIR, "failed_tickers.txt")

# ══════════════════════════════════════════════════════════════
print(f"[OK] Import 완료")
print(f"[OK] DEFAULT_TICKER_LIST: {len(DEFAULT_TICKER_LIST):,}개")
print(f"[설정] TABLE={TABLE_RESULT}  STORE_MODE={STORE_MODE}")
print(f"[설정] BETA_WINDOWS={BETA_WINDOWS}")
print(f"[설정] MIN_START_DATE={MIN_START_DATE}")


[OK] Import 완료
[OK] DEFAULT_TICKER_LIST: 2,000개
[설정] TABLE=us_required_return_result  STORE_MODE=minimal
[설정] BETA_WINDOWS=[252, 750, 1250]
[설정] MIN_START_DATE=2015-01-01


## Cell 3 · DB 연결 함수 / 조회 함수

In [3]:
# ── DB 접속 정보 ──────────────────────────────────────────────
db_info = get_db_info()   # config.py 의 get_db_info()


def get_conn(db_info: Dict[str, Any]):
    """pymysql 연결 생성"""
    return pymysql.connect(
        host        = db_info["host"],
        port        = db_info.get("port", DEFAULT_PORT),
        user        = db_info["user"],
        password    = db_info["password"],
        db          = db_info.get("database", DB_NAME),
        charset     = "utf8mb4",
        autocommit  = False,
        cursorclass = pymysql.cursors.DictCursor,
    )


def get_minmax_date_in_db(
    db_info: Dict[str, Any],
    ticker: str,
    indicator: str,
) -> Tuple[Optional[pd.Timestamp], Optional[pd.Timestamp]]:
    """MIN/MAX 날짜를 1회 쿼리로 조회 (커넥션 수 최소화)"""
    sql = f"""
        SELECT MIN(date) AS first_date, MAX(date) AS last_date
        FROM   {TABLE_RESULT}
        WHERE  ticker=%s AND indicator=%s;
    """
    conn = get_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (ticker, indicator))
            row = cur.fetchone()
            first_dt = row["first_date"] if row else None
            last_dt  = row["last_date"]  if row else None
    finally:
        conn.close()

    first_dt = pd.to_datetime(first_dt) if first_dt is not None else None
    last_dt  = pd.to_datetime(last_dt)  if last_dt  is not None else None
    return first_dt, last_dt


def read_indicator_series(
    db_info: Dict[str, Any],
    ticker: str,
    indicator: str,
    start_date: Optional[str] = None,
    end_date: Optional[str]   = None,
) -> pd.DataFrame:
    """DB에서 (date, value) 시리즈를 필요 구간만 읽어 반환"""
    where  = ["ticker=%s", "indicator=%s"]
    params = [ticker, indicator]
    if start_date is not None:
        where.append("date >= %s")
        params.append(start_date)
    if end_date is not None:
        where.append("date <= %s")
        params.append(end_date)

    sql = f"""
        SELECT date, value
        FROM   {TABLE_RESULT}
        WHERE  {" AND ".join(where)}
        ORDER  BY date;
    """
    conn = get_conn(db_info)
    try:
        df = pd.read_sql(sql, conn, params=params)
    finally:
        conn.close()

    if df.empty:
        return pd.DataFrame(columns=["date", "value"])

    df["date"]  = pd.to_datetime(df["date"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna(subset=["date"]).drop_duplicates("date").sort_values("date")
    return df


# ── 연결 테스트 ───────────────────────────────────────────────
try:
    _c = get_conn(db_info)
    with _c.cursor() as _cur:
        _cur.execute("SELECT 1")
    _c.close()
    log("DB", f"연결 성공  host={db_info.get('host')}  port={db_info.get('port')}  db={db_info.get('database')}")
except Exception as _e:
    log("DB", f"연결 실패: {_e}")


[21:38:54][DB] 연결 성공  host=192.168.0.230  port=3307  db=investar


## Cell 4 · FMP / FDR 가격 fetch 함수

In [4]:
def _get_json(url: str, params: Dict[str, Any]) -> Any:
    """FMP API 호출 (재시도 포함)"""
    last_err = None
    for k in range(MAX_RETRY):
        try:
            r = requests.get(url, params=params, timeout=30)
            if r.status_code == 429:
                time.sleep(1.0 + 0.7 * k)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(0.7 + 0.7 * k)
    raise RuntimeError(f"FMP 요청 실패 (재시도 {MAX_RETRY}회): {last_err}")


def fetch_fmp_price(
    symbol: str,
    api_key: str,
    start_date: str,
    end_date: Optional[str]   = None,
    min_start_date: str       = MIN_START_DATE,
) -> pd.DataFrame:
    """
    FMP에서 adjClose 우선으로 가격 fetch.
    start_date 가 min_start_date 보다 최근이면 min_start_date 로 당김.
    반환: columns [date, price]
    """
    if pd.to_datetime(start_date) > pd.to_datetime(min_start_date):
        start_date = min_start_date

    url    = f"https://financialmodelingprep.com/api/v3/historical-price-full/{symbol}"
    params = {"from": start_date, "apikey": api_key}
    if end_date is not None:
        params["to"] = end_date

    try:
        js   = _get_json(url, params=params)
        hist = js.get("historical", []) if isinstance(js, dict) else []
    except Exception:
        return pd.DataFrame(columns=["date", "price"])

    if not hist:
        return pd.DataFrame(columns=["date", "price"])

    df = pd.DataFrame(hist)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    price_col = "adjClose" if "adjClose" in df.columns else "close"
    df[price_col] = pd.to_numeric(df[price_col], errors="coerce")
    df = (
        df[["date", price_col]]
        .rename(columns={price_col: "price"})
        .dropna(subset=["price"])
        .drop_duplicates("date")
        .sort_values("date")
    )
    return df


def fetch_fdr_price(
    symbol: str,
    start_date: str,
    end_date: str,
) -> pd.DataFrame:
    """FinanceDataReader fallback. 반환: columns [date, price]"""
    try:
        df = fdr.DataReader(symbol, start_date, end_date)
    except Exception:
        return pd.DataFrame(columns=["date", "price"])

    if df is None or df.empty:
        return pd.DataFrame(columns=["date", "price"])

    df = df.copy().reset_index()
    # 날짜 컬럼 통일
    date_col = next((c for c in df.columns if c.lower() in ("date", "index")), df.columns[0])
    df = df.rename(columns={date_col: "date"})

    # 가격 컬럼 우선순위
    for pc in ("Adj Close", "adj close", "Close", "close"):
        if pc in df.columns:
            price_col = pc
            break
    else:
        num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if not num_cols:
            return pd.DataFrame(columns=["date", "price"])
        price_col = num_cols[0]

    out = df[["date", price_col]].rename(columns={price_col: "price"})
    out["date"]  = pd.to_datetime(out["date"], errors="coerce")
    out["price"] = pd.to_numeric(out["price"], errors="coerce")
    out = out.dropna(subset=["date", "price"]).drop_duplicates("date").sort_values("date")
    return out


print("[OK] fetch_fmp_price / fetch_fdr_price 함수 정의 완료")


[OK] fetch_fmp_price / fetch_fdr_price 함수 정의 완료


## Cell 5 · RF(국채금리) fetch 함수

In [5]:
def fetch_us_treasury_yields(
    start_date: str,
    end_date: Optional[str] = None,
) -> pd.DataFrame:
    """
    1y / 3y / 5y 미국 국채 수익률을 FRED 또는 Yahoo 로 fetch.
    반환: index=date, columns=[rf_1y, rf_3y, rf_5y]  (소수 단위, 예: 0.045)
    """
    if end_date is None:
        end_date = pd.Timestamp.today().strftime("%Y-%m-%d")

    # 1순위: FRED
    try:
        y1 = fdr.DataReader("FRED:DGS1", start_date, end_date)
        y3 = fdr.DataReader("FRED:DGS3", start_date, end_date)
        y5 = fdr.DataReader("FRED:DGS5", start_date, end_date)
        idx = y1.index.union(y3.index).union(y5.index)
        out = pd.DataFrame(index=idx).sort_index()
        out["rf_1y"] = pd.to_numeric(y1.iloc[:, 0], errors="coerce") / 100.0
        out["rf_3y"] = pd.to_numeric(y3.iloc[:, 0], errors="coerce") / 100.0
        out["rf_5y"] = pd.to_numeric(y5.iloc[:, 0], errors="coerce") / 100.0
        log("RF", f"FRED 로드 성공: {len(out)}행")
        return out
    except Exception:
        pass

    # 2순위: Yahoo (^IRX, ^FVX)
    try:
        y1 = fdr.DataReader("^IRX", start_date, end_date)
        y5 = fdr.DataReader("^FVX", start_date, end_date)
        idx = y1.index.union(y5.index)
        out = pd.DataFrame(index=idx).sort_index()
        out["rf_1y"] = pd.to_numeric(y1["Close"], errors="coerce") / 100.0
        out["rf_5y"] = pd.to_numeric(y5["Close"], errors="coerce") / 100.0
        out["rf_3y"] = out["rf_1y"] + (out["rf_5y"] - out["rf_1y"]) * (3 - 1) / (5 - 1)
        log("RF", f"Yahoo fallback 로드 성공: {len(out)}행")
        return out
    except Exception as e:
        log("RF", f"[WARN] 국채 금리 로드 실패: {e}")
        return pd.DataFrame(columns=["rf_1y", "rf_3y", "rf_5y"])


print("[OK] fetch_us_treasury_yields 함수 정의 완료")


[OK] fetch_us_treasury_yields 함수 정의 완료


## Cell 6 · Beta · Required Return 계산 함수

In [6]:
def rolling_beta(
    ret_stock: pd.Series,
    ret_mkt: pd.Series,
    window: int,
) -> pd.Series:
    """Rolling Beta = Cov(ret_stock, ret_mkt) / Var(ret_mkt)"""
    cov = ret_stock.rolling(window).cov(ret_mkt)
    var = ret_mkt.rolling(window).var()
    return cov / var


def build_features(
    price_stock_df: pd.DataFrame,   # columns: [date, price_stock]
    price_mkt_df: pd.DataFrame,     # columns: [date, price_mkt]
    rf_df: pd.DataFrame,            # index=date, columns=[rf_1y, rf_3y, rf_5y]
) -> pd.DataFrame:
    """
    Beta 및 Required Return (CAPM) 계산.

    저장 지표:
      minimal : beta_252 / beta_750 / beta_1250 / Re_1y / Re_3y / Re_5y
      full    : 위 + price_mkt / ret_stock / ret_mkt / rf_* / E_Rm_*
    """
    df = pd.merge(price_stock_df, price_mkt_df, on="date", how="inner")
    df = df.sort_values("date").drop_duplicates("date").reset_index(drop=True)

    df["price_stock"] = pd.to_numeric(df["price_stock"], errors="coerce")
    df["price_mkt"]   = pd.to_numeric(df["price_mkt"],   errors="coerce")
    df = df.dropna(subset=["price_stock", "price_mkt"])

    df["ret_stock"] = df["price_stock"].pct_change()
    df["ret_mkt"]   = df["price_mkt"].pct_change()

    # Rolling Beta
    for w in BETA_WINDOWS:
        df[f"beta_{w}"] = rolling_beta(df["ret_stock"], df["ret_mkt"], w)

    # RF 조인
    if rf_df is None or rf_df.empty:
        df["rf_1y"] = np.nan
        df["rf_3y"] = np.nan
        df["rf_5y"] = np.nan
    else:
        rf2 = rf_df.copy().sort_index()
        rf2 = rf2.reindex(pd.to_datetime(df["date"])).ffill()
        df["rf_1y"] = rf2["rf_1y"].values
        df["rf_3y"] = rf2["rf_3y"].values
        df["rf_5y"] = rf2["rf_5y"].values

    # 기대 시장 수익률 (연환산)
    df["E_Rm_1y"] = df["ret_mkt"].rolling(252).mean()  * 252
    df["E_Rm_3y"] = df["ret_mkt"].rolling(750).mean()  * 252
    df["E_Rm_5y"] = df["ret_mkt"].rolling(1250).mean() * 252

    # CAPM Required Return
    df["Re_1y"] = df["rf_1y"] + df["beta_252"]  * (df["E_Rm_1y"] - df["rf_1y"])
    df["Re_3y"] = df["rf_3y"] + df["beta_750"]  * (df["E_Rm_3y"] - df["rf_3y"])
    df["Re_5y"] = df["rf_5y"] + df["beta_1250"] * (df["E_Rm_5y"] - df["rf_5y"])

    return df


print("[OK] rolling_beta / build_features 함수 정의 완료")
print(f"     BETA_WINDOWS = {BETA_WINDOWS}")
print(f"     저장 지표 (minimal): beta_252/750/1250, Re_1y/3y/5y")


[OK] rolling_beta / build_features 함수 정의 완료
     BETA_WINDOWS = [252, 750, 1250]
     저장 지표 (minimal): beta_252/750/1250, Re_1y/3y/5y


## Cell 7 · MySQL 저장 함수 (sanitize + upsert)

- NaN / inf → `None` 변환 후 저장
- `INSERT ... ON DUPLICATE KEY UPDATE value = VALUES(value)`
- ticker 단위로 커넥션 분리 → 메모리/커넥션 부담 최소화

In [7]:
def _to_mysql_float(x: Any) -> Optional[float]:
    """NaN / inf → None, 그 외 float 반환"""
    if x is None:
        return None
    try:
        v = float(x)
    except Exception:
        return None
    return None if (math.isnan(v) or math.isinf(v)) else v


def sanitize_long_for_mysql(df: pd.DataFrame) -> pd.DataFrame:
    """저장 전 데이터 정제"""
    out = df.copy()
    out["date"]      = pd.to_datetime(out["date"], errors="coerce")
    out = out[out["date"].notna()]
    out["date"]      = out["date"].dt.date
    out["ticker"]    = out["ticker"].astype(str)
    out["indicator"] = out["indicator"].astype(str)
    out = out[~out["ticker"].str.lower().isin(["nan", "none"])]
    out = out[~out["indicator"].str.lower().isin(["nan", "none"])]
    out["value"]     = pd.to_numeric(out["value"], errors="coerce")
    out["value"]     = out["value"].apply(_to_mysql_float)
    out = out.drop_duplicates(subset=["date", "ticker", "indicator"])
    return out


def upsert_long_df(
    db_info: Dict[str, Any],
    long_df: pd.DataFrame,
    batch_size_rows: int   = 50_000,
    batch_size_ticker: int = 50,
    drop_null_values: bool = True,
) -> int:
    """
    long_df → DB upsert.
    PK (date, ticker, indicator) 중복 시 value 업데이트.
    Returns: 저장 행 수
    """
    if long_df is None or long_df.empty:
        return 0

    df = sanitize_long_for_mysql(long_df)
    if drop_null_values:
        df = df.dropna(subset=["value"])
    if df.empty:
        return 0

    tickers = sorted(df["ticker"].unique())
    total_saved = 0

    insert_sql = f"""
        INSERT INTO {TABLE_RESULT} (date, ticker, indicator, value)
        VALUES (%s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            value = VALUES(value);
    """

    for i in range(0, len(tickers), batch_size_ticker):
        batch_tickers = tickers[i : i + batch_size_ticker]
        batch = (
            df[df["ticker"].isin(batch_tickers)]
            .sort_values(["ticker", "date", "indicator"])
        )
        rows = list(batch[["date", "ticker", "indicator", "value"]].itertuples(index=False, name=None))

        conn = get_conn(db_info)
        try:
            with conn.cursor() as cur:
                for j in range(0, len(rows), batch_size_rows):
                    chunk = rows[j : j + batch_size_rows]
                    # NaN/inf 샘플 검사
                    for _r in chunk[:10]:
                        vv = _r[3]
                        if isinstance(vv, float) and (math.isnan(vv) or math.isinf(vv)):
                            raise ValueError(f"NaN/inf 발견: {_r}")
                    cur.executemany(insert_sql, chunk)
            conn.commit()
            log("DB", f"batch {i//batch_size_ticker+1}: tickers={len(batch_tickers)}, rows={len(rows):,}")
            total_saved += len(rows)
        except Exception:
            conn.rollback()
            raise
        finally:
            conn.close()

    return total_saved


# ── 체크포인트 유틸 ───────────────────────────────────────────
def _load_set(path: str) -> set:
    if not os.path.exists(path):
        return set()
    with open(path, "r", encoding="utf-8") as f:
        return {line.strip() for line in f if line.strip()}

def _append_line(path: str, line: str) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(line.strip() + "\n")


print("[OK] sanitize_long_for_mysql / upsert_long_df 함수 정의 완료")


[OK] sanitize_long_for_mysql / upsert_long_df 함수 정의 완료


## Cell 8 · 가격 확보 함수 (ensure_price_series)

- DB에 있는 구간 확인 → **부족분만** FMP 또는 FDR 호출
- backfill (과거) / forward (최신) 각각 필요한 구간만 요청
- FMP 실패 시 FDR 자동 fallback

In [8]:
def ensure_price_series(
    db_info: Dict[str, Any],
    ticker: str,
    api_key: str,
    indicator_name: str       = "price_stock",
    today_iso: Optional[str]  = None,
    min_start_date: str       = MIN_START_DATE,
    return_start_date: Optional[str] = None,
) -> pd.DataFrame:
    """
    DB에서 가격을 확인하고 부족분만 FMP/FDR 로 fetch 후 저장.

    Returns
    -------
    DataFrame  columns: [date, {indicator_name}]
    """
    if today_iso is None:
        today_iso = datetime.utcnow().date().isoformat()

    first_dt, last_dt = get_minmax_date_in_db(db_info, ticker, indicator_name)

    def _fetch(start: str, end: Optional[str]) -> pd.DataFrame:
        """FMP 먼저, 실패 시 FDR fallback"""
        df = fetch_fmp_price(ticker, api_key, start, end, min_start_date)
        time.sleep(SLEEP_BETWEEN_CALLS)
        if df.empty:
            df = fetch_fdr_price(ticker, start, end or today_iso)
        return df

    def _store(df_price: pd.DataFrame):
        if df_price is None or df_price.empty:
            return
        df2 = df_price.rename(columns={"price": indicator_name}).copy()
        long_new = df2.assign(ticker=ticker).melt(
            id_vars=["date", "ticker"],
            value_vars=[indicator_name],
            var_name="indicator",
            value_name="value",
        )
        upsert_long_df(db_info, long_new, drop_null_values=True)

    # (A) DB에 데이터 없음 → 전 구간 fetch
    if last_dt is None:
        df_new = _fetch(min_start_date, today_iso)
        if not df_new.empty:
            _store(df_new)
        out = df_new.rename(columns={"price": indicator_name}).drop_duplicates("date").sort_values("date")
        if return_start_date:
            out = out[out["date"] >= pd.to_datetime(return_start_date)]
        return out

    # (B) 과거 backfill — min_start_date 보다 늦게 시작한 경우
    if first_dt is not None and first_dt.date().isoformat() > min_start_date:
        back_end = (first_dt - pd.Timedelta(days=1)).date().isoformat()
        df_back  = _fetch(min_start_date, back_end)
        if not df_back.empty:
            _store(df_back)

    # (C) 최신 forward
    start_missing = (last_dt + pd.Timedelta(days=1)).date().isoformat()
    if start_missing <= today_iso:
        df_fwd = _fetch(start_missing, today_iso)
        if not df_fwd.empty:
            _store(df_fwd)

    # (D) 필요 구간만 DB에서 읽어 반환
    df_db = read_indicator_series(
        db_info, ticker, indicator_name,
        start_date=return_start_date, end_date=today_iso,
    ).rename(columns={"value": indicator_name})
    df_db["date"] = pd.to_datetime(df_db["date"], errors="coerce")
    df_db = df_db.dropna(subset=["date"]).drop_duplicates("date").sort_values("date")
    return df_db


print("[OK] ensure_price_series 함수 정의 완료")


[OK] ensure_price_series 함수 정의 완료


## Cell 9 · 티커 1개 업데이트 함수 (update_one_ticker)

- `last_saved_date` 이후 데이터만 저장 (중복 저장 방지)
- `price_stock` 은 `ensure_price_series` 에서 이미 저장 → 재저장 금지

In [9]:
def _pick_ref_indicator(store_mode: str) -> str:
    """저장 여부 판단 기준 지표 (Re_5y 가 가장 보수적)"""
    return "Re_5y"


def update_one_ticker(
    db_info: Dict[str, Any],
    ticker: str,
    api_key: str,
    spy_price_df: pd.DataFrame,   # columns: [date, price_mkt]
    rf_df: pd.DataFrame,          # index=date, columns=[rf_1y, rf_3y, rf_5y]
    today_iso: Optional[str]  = None,
    min_start_date: str       = MIN_START_DATE,
    store_mode: str           = "minimal",
) -> Tuple[bool, str]:
    """
    티커 1개에 대해 가격 확보 → 계산 → DB 저장 수행.

    Returns
    -------
    (True, 성공 메시지) or (False, 실패 메시지)
    """
    if today_iso is None:
        today_iso = datetime.utcnow().date().isoformat()

    if spy_price_df is None or spy_price_df.empty:
        return (False, "SPY price df 없음")

    # 0) 마지막 저장일 확인 → 중복 계산/저장 방지
    ref_ind          = _pick_ref_indicator(store_mode)
    _, last_saved    = get_minmax_date_in_db(db_info, ticker, ref_ind)

    if last_saved is not None:
        calc_start_dt = (pd.to_datetime(last_saved) - BDay(ROLLING_WARMUP_BDAYS)).date().isoformat()
        save_after_dt = pd.to_datetime(last_saved).date()
    else:
        calc_start_dt = min_start_date
        save_after_dt = None

    # 1) 종목 가격 확보 (필요 구간만 FMP fetch) + DB 저장
    px_stock = ensure_price_series(
        db_info           = db_info,
        ticker            = ticker,
        api_key           = api_key,
        indicator_name    = "price_stock",
        today_iso         = today_iso,
        min_start_date    = min_start_date,
        return_start_date = calc_start_dt,
    )
    if px_stock.empty:
        return (False, f"{ticker}: price_stock empty")

    # 2) SPY 가격을 calc_start_dt 이후 구간만 슬라이싱
    spy2 = spy_price_df.copy()
    spy2["date"] = pd.to_datetime(spy2["date"], errors="coerce")
    spy2 = spy2.dropna(subset=["date", "price_mkt"])
    spy2 = spy2[spy2["date"] >= pd.to_datetime(calc_start_dt)]
    if spy2.empty:
        return (False, f"{ticker}: SPY slice empty from {calc_start_dt}")

    # 3) 계산
    feat = build_features(
        price_stock_df = px_stock[["date", "price_stock"]],
        price_mkt_df   = spy2[["date", "price_mkt"]],
        rf_df          = rf_df,
    )
    if feat.empty:
        return (False, f"{ticker}: feature df empty")

    feat["ticker"] = ticker

    # 4) 저장 컬럼 선택
    if store_mode == "full":
        keep_cols = [
            # price_stock 은 ensure_price_series 에서 이미 저장 → 제외
            "price_mkt", "ret_stock", "ret_mkt",
            "beta_252", "beta_750", "beta_1250",
            "rf_1y", "rf_3y", "rf_5y",
            "E_Rm_1y", "E_Rm_3y", "E_Rm_5y",
            "Re_1y", "Re_3y", "Re_5y",
        ]
    else:  # minimal
        keep_cols = [
            "beta_252", "beta_750", "beta_1250",
            "Re_1y", "Re_3y", "Re_5y",
        ]
    keep_cols = [c for c in keep_cols if c in feat.columns]

    # 5) last_saved 이후만 저장 → 중복 upsert 방지
    if save_after_dt is not None:
        feat = feat[pd.to_datetime(feat["date"]).dt.date > save_after_dt]
        if feat.empty:
            return (True, f"{ticker}: up-to-date (no new rows after {save_after_dt})")

    # 6) wide → long melt
    long_df = feat[["date", "ticker"] + keep_cols].melt(
        id_vars    = ["date", "ticker"],
        value_vars = keep_cols,
        var_name   = "indicator",
        value_name = "value",
    )
    long_df["value"] = pd.to_numeric(long_df["value"], errors="coerce")
    long_df = long_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["value"])

    if long_df.empty:
        return (False, f"{ticker}: 모든 값 NaN")

    # 7) DB 저장
    n_saved = upsert_long_df(db_info, long_df, drop_null_values=True)
    return (True, f"{ticker}: saved {n_saved:,}행 ({store_mode}) from {calc_start_dt}")


print("[OK] update_one_ticker 함수 정의 완료")


[OK] update_one_ticker 함수 정의 완료


## Cell 10 · SPY / RF 공통 데이터 준비

SPY 가격과 RF(국채금리)는 배치 전 1회만 fetch해서 모든 티커에 재사용합니다.

In [10]:
def prepare_spy_and_rf(
    db_info: Dict[str, Any],
    api_key: str,
    today_iso: str,
    min_start_date: str = MIN_START_DATE,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    SPY 가격 + RF 를 1회 fetch 후 반환.
    SPY 가격은 DB에도 저장 (ensure_price_series 경유).

    Returns
    -------
    (spy_price_df, rf_df)
      spy_price_df : columns [date, price_mkt]
      rf_df        : index=date, columns=[rf_1y, rf_3y, rf_5y]
    """
    log("PREP", "SPY 가격 확보 시작")
    spy_px = ensure_price_series(
        db_info           = db_info,
        ticker            = MARKET_TICKER,
        api_key           = api_key,
        indicator_name    = "price_stock",
        today_iso         = today_iso,
        min_start_date    = min_start_date,
        return_start_date = min_start_date,
    )

    # FDR 최종 안전망
    if spy_px.empty:
        log("PREP", "[WARN] SPY ensure 실패 → FDR 직접 fallback")
        df_fdr = fetch_fdr_price(MARKET_TICKER, min_start_date, today_iso)
        if not df_fdr.empty:
            df2 = df_fdr.rename(columns={"price": "price_stock"})
            long_new = df2.assign(ticker=MARKET_TICKER).melt(
                id_vars=["date", "ticker"], value_vars=["price_stock"],
                var_name="indicator", value_name="value",
            )
            upsert_long_df(db_info, long_new, drop_null_values=True)
            spy_px = df2[["date", "price_stock"]].copy()

    if spy_px.empty:
        raise RuntimeError("SPY 가격 확보 실패. API_KEY / 네트워크 / FDR 상태를 확인하세요.")

    spy_price_df = (
        spy_px
        .rename(columns={"price_stock": "price_mkt"})
        [["date", "price_mkt"]]
        .copy()
    )
    spy_price_df["date"]      = pd.to_datetime(spy_price_df["date"])
    spy_price_df["price_mkt"] = pd.to_numeric(spy_price_df["price_mkt"], errors="coerce")
    spy_price_df = (
        spy_price_df
        .dropna(subset=["date", "price_mkt"])
        .drop_duplicates("date")
        .sort_values("date")
    )
    log("PREP", f"SPY 가격: {len(spy_price_df)}행  ({spy_price_df['date'].iloc[0].date()} ~ {spy_price_df['date'].iloc[-1].date()})")

    # RF
    start_for_rf = spy_price_df["date"].min().date().isoformat()
    end_for_rf   = spy_price_df["date"].max().date().isoformat()
    log("PREP", f"RF 로드 중: {start_for_rf} ~ {end_for_rf}")
    rf_df = fetch_us_treasury_yields(start_for_rf, end_for_rf)
    if rf_df is None or rf_df.empty:
        log("PREP", "[WARN] RF 로드 실패 → Required Return 이 NaN 이 될 수 있습니다.")

    return spy_price_df, rf_df


print("[OK] prepare_spy_and_rf 함수 정의 완료")


[OK] prepare_spy_and_rf 함수 정의 완료


## Cell 11 · 배치 실행

### 실행 모드
| 변수 | 설명 |
|------|------|
| `RUN_TICKERS` | `None` → 구간/전체 실행 / 리스트 → 해당 티커만 |
| `TICKER_START` | 리스트 슬라이싱 시작 인덱스 (0부터) |
| `TICKER_END` | 슬라이싱 끝 인덱스 (None = 끝까지) |
| `SKIP_DONE` | True → 체크포인트에 있는 티커 건너뜀 |
| `RETRY_FAILED` | True → 실패 티커 1회 재시도 |

### 구간 예시
```python
TICKER_START, TICKER_END = 0,    500   # 0~499번째
TICKER_START, TICKER_END = 500,  1000  # 500~999번째
TICKER_START, TICKER_END = None, None  # 전체
```

### 메모리 전략
> 1개 처리 → 즉시 저장 → `gc.collect()` → 다음 티커  
> 완료 티커는 `done_tickers.txt` 에 기록 → 재시작 시 중복 처리 방지

In [11]:
# ══════════════════════════════════════════════════════════════
#  배치 설정 — 여기를 수정하세요
# ══════════════════════════════════════════════════════════════

RUN_TICKERS  = None          # type: Optional[List[str]]
# RUN_TICKERS = ["AAPL", "MSFT", "NVDA"]   # 특정 티커만

TICKER_START = 0             # type: Optional[int]
TICKER_END   = 20           # type: Optional[int]
# 전체: TICKER_START = None, TICKER_END = None

RUN_STORE_MODE   = STORE_MODE   # "minimal" or "full"
SKIP_DONE        = True          # True: 체크포인트 티커 스킵
RETRY_FAILED     = True          # True: 실패 티커 1회 재시도

DONE_PATH = DEFAULT_DONE_PATH
FAIL_PATH = DEFAULT_FAIL_PATH

# ══════════════════════════════════════════════════════════════
#  실행 대상 결정
# ══════════════════════════════════════════════════════════════
if RUN_TICKERS is not None:
    tickers = RUN_TICKERS
    log("BATCH", f"모드: 특정 티커 지정  {tickers}")
else:
    tickers = DEFAULT_TICKER_LIST[TICKER_START:TICKER_END]
    _s = TICKER_START if TICKER_START is not None else 0
    _e = TICKER_END   if TICKER_END   is not None else len(DEFAULT_TICKER_LIST)
    log("BATCH", f"모드: 구간 실행  index {_s} ~ {_e-1}  ({len(tickers)}개)")

total     = len(tickers)
today_iso = datetime.utcnow().date().isoformat()
done_set  = _load_set(DONE_PATH) if SKIP_DONE else set()
fail_set  = set()

# ── SPY / RF 공통 데이터 준비 (배치 전 1회) ──────────────────
spy_price_df, rf_df = prepare_spy_and_rf(
    db_info        = db_info,
    api_key        = API_KEY,
    today_iso      = today_iso,
    min_start_date = MIN_START_DATE,
)

success, skipped, errored = 0, 0, 0

log("BATCH", "=" * 70)
log("BATCH", f"시작  | {total}개 티커 | store_mode={RUN_STORE_MODE}")
log("BATCH", f"skip_done={SKIP_DONE}  retry_failed={RETRY_FAILED}")
log("BATCH", f"체크포인트 완료: {len(done_set)}개")
log("BATCH", "=" * 70)


def _run_one(ticker: str) -> Tuple[bool, str]:
    """티커 1개 처리. (True, msg) or (False, msg) 반환."""
    try:
        ok, msg = update_one_ticker(
            db_info        = db_info,
            ticker         = ticker,
            api_key        = API_KEY,
            spy_price_df   = spy_price_df,
            rf_df          = rf_df,
            today_iso      = today_iso,
            min_start_date = MIN_START_DATE,
            store_mode     = RUN_STORE_MODE,
        )
    except Exception as e:
        return (False, f"{ticker}: 예외 발생 → {e}")
    finally:
        gc.collect()
    return ok, msg


# ── 메인 루프 ─────────────────────────────────────────────────
for idx, ticker in enumerate(tickers, 1):
    ticker = str(ticker).strip()
    if not ticker:
        continue

    pct = idx / total * 100
    log("PROGRESS", f"[{idx:>4}/{total}] ({pct:5.1f}%)  >>  {ticker}")

    if SKIP_DONE and ticker in done_set:
        log(ticker, "[SKIP] 이미 완료 (체크포인트)")
        skipped += 1
        continue

    ok, msg = _run_one(ticker)
    if ok:
        log(ticker, f"[OK] {msg}")
        _append_line(DONE_PATH, ticker)
        done_set.add(ticker)
        success += 1
    else:
        log(ticker, f"[FAIL] {msg}")
        _append_line(FAIL_PATH, ticker)
        fail_set.add(ticker)
        errored += 1

# ── 실패 티커 재시도 ──────────────────────────────────────────
if RETRY_FAILED and fail_set:
    log("RETRY", f"실패 티커 {len(fail_set)}개 재시도")
    still_fail = set()
    for ticker in sorted(fail_set):
        ok, msg = _run_one(ticker)
        if ok:
            log(ticker, f"[RETRY-OK] {msg}")
            _append_line(DONE_PATH, ticker)
            done_set.add(ticker)
            success += 1
            errored -= 1
        else:
            log(ticker, f"[RETRY-FAIL] {msg}")
            still_fail.add(ticker)
    log("RETRY", f"재시도 완료. 최종 실패: {len(still_fail)}개")

# ── 배치 요약 ─────────────────────────────────────────────────
log("BATCH", "=" * 70)
log("BATCH", f"완료 | 성공={success}  스킵={skipped}  실패={errored}  합계={total}")
log("BATCH", f"체크포인트: {DONE_PATH}")
log("BATCH", "=" * 70)


[21:38:54][BATCH] 모드: 구간 실행  index 0 ~ 19  (20개)
[21:38:54][PREP] SPY 가격 확보 시작
[21:39:12][DB] batch 1: tickers=1, rows=2,823
[21:39:31][PREP] [WARN] SPY ensure 실패 → FDR 직접 fallback
[21:39:33][DB] batch 1: tickers=1, rows=2,824
[21:39:33][PREP] SPY 가격: 2824행  (2014-12-31 ~ 2026-03-25)
[21:39:33][PREP] RF 로드 중: 2014-12-31 ~ 2026-03-25
[21:39:35][RF] Yahoo fallback 로드 성공: 2931행
[21:39:35][BATCH] ======================================================================
[21:39:35][BATCH] 시작  | 20개 티커 | store_mode=minimal
[21:39:35][BATCH] skip_done=True  retry_failed=True
[21:39:35][BATCH] 체크포인트 완료: 3575개
[21:39:35][BATCH] ======================================================================
[21:39:35][PROGRESS] [   1/20] (  5.0%)  >>  NVDA
[21:40:07][DB] batch 1: tickers=1, rows=2,823
[21:40:17][NVDA] [FAIL] NVDA: price_stock empty
[21:40:17][PROGRESS] [   2/20] ( 10.0%)  >>  GOOG
[21:40:17][GOOG] [SKIP] 이미 완료 (체크포인트)
[21:40:17][PROGRESS] [   3/20] ( 15.0%)  >>  AAPL
[21:40:49][DB] batch 1: 

## Cell 12 · DB 조회 유틸 (pivot 조회)

long-format → pivot 형태로 변환해 반환합니다.

In [14]:
def fetch_required_return_pivot_from_db(
    db_info: Dict[str, Any],
    table_name: str               = TABLE_RESULT,
    indicators: Optional[List[str]] = None,
    start_date: Optional[str]     = None,
    end_date: Optional[str]       = None,
    tickers: Optional[List[str]]  = None,
) -> pd.DataFrame:
    """
    DB long-format → pivot DataFrame.
    반환 컬럼: date, ticker, [indicator 컬럼들...]

    pd.read_sql + pymysql 조합의 컬럼명-데이터 혼입 버그를 피하기 위해
    pymysql 커서로 직접 조회 후 pd.DataFrame 으로 변환합니다.
    """
    if indicators is None:
        indicators = ["Re_1y", "Re_3y", "Re_5y",
                      "beta_252", "beta_750", "beta_1250"]

    # ── CASE WHEN: indicator 값을 SQL 리터럴로 직접 삽입 ────────
    case_parts = []
    for ind in indicators:
        ind_esc = ind.replace("'", "''").replace("`", "``")
        case_parts.append(
            f"MAX(CASE WHEN indicator = '{ind_esc}' THEN value END) AS `{ind_esc}`"
        )
    case_sql = ",\n                ".join(case_parts)

    # ── WHERE 절 ─────────────────────────────────────────────────
    where_clauses = ["date != 'date'"]   # 오염 행 방어
    params        = []

    if start_date:
        where_clauses.append("date >= %s")
        params.append(start_date)
    if end_date:
        where_clauses.append("date <= %s")
        params.append(end_date)
    if tickers:
        placeholders = ", ".join(["%s"] * len(tickers))
        where_clauses.append(f"ticker IN ({placeholders})")
        params.extend(tickers)

    where_sql = "WHERE " + " AND ".join(where_clauses)

    query = f"""
        SELECT
            date,
            ticker,
            {case_sql}
        FROM   {table_name}
        {where_sql}
        GROUP  BY date, ticker
        ORDER  BY date, ticker;
    """

    # ── pymysql 커서로 직접 조회 → DataFrame 변환 ───────────────
    # pd.read_sql + pymysql 조합은 컬럼명을 첫 행으로 반환하는 버그 있음
    conn = get_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(query, params if params else None)
            rows = cur.fetchall()   # List[Dict]
    finally:
        conn.close()

    if not rows:
        cols = ["date", "ticker"] + indicators
        return pd.DataFrame(columns=cols)

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"]).reset_index(drop=True)

    # value 컬럼 숫자 변환
    for col in indicators:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


print("[OK] fetch_required_return_pivot_from_db 재정의 완료")
print("     pymysql 커서 직접 조회 방식 적용 (pd.read_sql 버그 우회)")


[OK] fetch_required_return_pivot_from_db 재정의 완료
     pymysql 커서 직접 조회 방식 적용 (pd.read_sql 버그 우회)


## Cell 13 · 결과 조회 테스트

In [17]:
# ── 단일 티커 조회 테스트 ─────────────────────────────────────
TEST_TICKER = "AAPL"   # ← 변경 가능

re_df = fetch_required_return_pivot_from_db(
    db_info    = db_info,
    table_name = TABLE_RESULT,
    indicators = ["Re_1y", "Re_3y", "Re_5y", "beta_252", "beta_750", "beta_1250"],
    start_date = "2020-01-01",
    end_date   = None,
    tickers    = [TEST_TICKER],
)

print(f"[조회] {TEST_TICKER}  총 {len(re_df)}행")
print("\n--- 앞 5행 ---")
display(re_df.head())
print("\n--- 뒤 5행 ---")
display(re_df.tail(10))


[조회] AAPL  총 1565행

--- 앞 5행 ---


,date,ticker,Re_1y,Re_3y,Re_5y,beta_252,beta_750,beta_1250
0,2020-01-02,AAPL,0.439061,0.197742,0.152094,1.553092,1.369364,1.241175
1,2020-01-03,AAPL,0.433768,0.194447,0.152494,1.447101,1.369372,1.239492
2,2020-01-06,AAPL,0.394956,0.194831,0.150513,1.463324,1.369466,1.243087
3,2020-01-07,AAPL,0.381275,0.194678,0.149214,1.471397,1.369474,1.242719
4,2020-01-08,AAPL,0.375207,0.196122,0.149212,1.471053,1.370458,1.243174



--- 뒤 5행 ---


,date,ticker,Re_1y,Re_3y,Re_5y,beta_252,beta_750,beta_1250
1555,2026-03-12,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1556,2026-03-13,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1557,2026-03-16,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1558,2026-03-17,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1559,2026-03-18,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1560,2026-03-19,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1561,2026-03-20,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1562,2026-03-23,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1563,2026-03-24,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1564,2026-03-25,AAPL,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
re_df

,date,ticker,Re_1y,Re_3y,Re_5y,beta_252,beta_750,beta_1250
0,2020-01-02,AAPL,0.439061,0.197742,0.152094,1.553092,1.369364,1.241175
1,2020-01-03,AAPL,0.433768,0.194447,0.152494,1.447101,1.369372,1.239492
2,2020-01-06,AAPL,0.394956,0.194831,0.150513,1.463324,1.369466,1.243087
3,2020-01-07,AAPL,0.381275,0.194678,0.149214,1.471397,1.369474,1.242719
4,2020-01-08,AAPL,0.375207,0.196122,0.149212,1.471053,1.370458,1.243174
...,...,...,...,...,...,...,...,...
1560,2026-03-19,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1561,2026-03-20,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1562,2026-03-23,AAPL,NaN,NaN,NaN,NaN,NaN,NaN
1563,2026-03-24,AAPL,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ── 전체 저장 현황 요약 (커서 직접 조회) ────────────────────────
conn = get_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT
                indicator,
                COUNT(DISTINCT ticker) AS ticker_cnt,
                COUNT(*)               AS total_rows,
                MIN(date)              AS date_from,
                MAX(date)              AS date_to
            FROM   {TABLE_RESULT}
            WHERE  date != 'date'
            GROUP  BY indicator
            ORDER  BY indicator;
        """)
        rows = cur.fetchall()
finally:
    conn.close()

summary_df = pd.DataFrame(rows)
print(f"[전체 저장 현황]  {TABLE_RESULT}")
display(summary_df)
